# TP3C · Experimentos

Este notebook corre los experimentos sobre el simulador de `simulador.ipynb` y deja los resultados en `resultados/`. Carga sólo las celdas del simulador, y si una corrida se corta, retoma: lo que ya está guardado no se vuelve a correr.

Se corre en dos rondas. La primera da 30 réplicas de un mes a cada alternativa, para descartar. La segunda da a las que quedan el N calculado. Con 30 réplicas el intervalo es más ancho que con N, y ese costo se acepta porque baja el cómputo de las alternativas que no van a quedar.

In [1]:
import json, multiprocessing as mp, os, time
for celda in json.load(open("simulador.ipynb"))["cells"]:
    if "modelo" in celda["metadata"].get("tags", []):
        exec("".join(celda["source"]), globals())
os.makedirs("resultados", exist_ok=True)
print("escala", P["escala"], "  factor de fin de semana", round(P["factor_finde"], 3))

escala 57.3   factor de fin de semana 1.178


## Ronda 1 · Una palanca por vez

Cada alternativa mueve una sola palanca y deja las demás como hoy. Todas usan las mismas semillas, para que cada réplica de una alternativa sea comparable con la misma réplica de hoy. La fricción se prueba con umbrales 5, 10 y 20 y reducciones 0,3 y 0,6.

El clasificador nuevo se prueba sólo con 2 moderadores, porque con uno la cola se desborda. Su comparación de referencia es la alternativa de 2 moderadores con el clasificador actual, además de hoy.

In [2]:
ALTERNATIVAS = {"hoy": dict(c=1), "c2": dict(c=2), "c3": dict(c=3)}
for u in (5, 10, 20):
    for r_ in (.3, .6):
        ALTERNATIVAS[f"friccion_u{u}_r{int(r_ * 10)}"] = dict(umbral=u, reduccion=r_)
for d_ in (.3, .5, .8):
    ALTERNATIVAS[f"degradacion_{int(d_ * 10)}"] = dict(degradar=True, d=d_)
for nombre, cl in (("clasificador_85_75", (.85, .75)), ("clasificador_95_90", (.95, .9))):
    ALTERNATIVAS[f"{nombre}_c2"] = dict(c=2, clasificador=cl)
SEMILLAS = list(range(100, 130))
DIAS, CALENTAMIENTO = 30, 2

def replica(a):
    nombre, vd, semilla, escenario = a
    t0 = time.time()
    r = RedBaires(P, vd, semilla=semilla, escenario=escenario).correr(dias=DIAS, calentamiento=CALENTAMIENTO)
    try:
        controles(r); ok = "ok"
    except AssertionError as e:
        ok = str(e)
    return dict(alternativa=nombre, semilla=semilla, **kpis(r), control=ok, segundos=time.time() - t0)

def log(archivo, texto):
    with open(archivo, "a") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S ") + texto + "\n")

def correr_ronda(carpeta, alternativas, semillas, escenario=None):
    """Una alternativa por vez, en 4 procesos. Se guarda al terminar cada una; si ya está completa, no se vuelve a correr."""
    os.makedirs(carpeta, exist_ok=True)
    salida_log = carpeta + ".log"
    with mp.get_context("fork").Pool(4) as pool:
        for nombre, vd in alternativas.items():
            ruta = f"{carpeta}/{nombre}.csv"
            if os.path.exists(ruta) and len(pd.read_csv(ruta)) == len(semillas):
                log(salida_log, f"{nombre}: ya estaba"); continue
            t0 = time.time()
            filas = pool.map(replica, [(nombre, vd, s, escenario) for s in semillas], chunksize=1)
            pd.DataFrame(filas).to_csv(ruta + ".tmp", index=False); os.replace(ruta + ".tmp", ruta)
            malos = sum(f["control"] != "ok" for f in filas)
            log(salida_log, f"{nombre}: {len(filas)} réplicas en {time.time() - t0:.0f} s, controles fallidos: {malos}")
    log(salida_log, "ronda terminada")

In [3]:
correr_ronda("resultados/ronda1", ALTERNATIVAS, SEMILLAS)
ronda1 = pd.concat([pd.read_csv(f"resultados/ronda1/{n}.csv") for n in ALTERNATIVAS])
assert len(ronda1) == len(ALTERNATIVAS) * len(SEMILLAS), "faltan réplicas"
assert (ronda1.control == "ok").all(), ronda1.loc[ronda1.control != "ok", ["alternativa", "semilla", "control"]].to_string()
print("réplicas:", len(ronda1), " controles: ok")

réplicas: 420  controles: ok
